# PJM OASIS CLI quickstart

Use this notebook to learn and verify the default PJM API path used by `pjm-api`: Python loads configuration, Java runs PJM's official `pjm-cli.jar`, and the jar sends authenticated OASIS requests.

The normal setup is persistent. Install the jar once at `~/.pjm/cli/pjm-cli.jar`, run `pjm-api init` once to create `~/.pjm/credentials.enc`, then scripts can import `CliBackend` and `load_settings` without hard-coding secrets.

Run the setup and local-check cells first. Start with TRAIN. Keep TEST and STAGE blank unless PJM gave you private URLs. Production reads are opt-in, and production write or reservation-style actions stay blocked by the package guard unless you explicitly allow them.


## 1. Import the package pieces

The notebook uses the same package code as the CLI. These imports show the real pieces: settings resolution, the Java CLI backend, the optional jar installer, doctor checks, command masking, result qualification, template metadata, and production guards.


In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path
from shutil import which

from pjm_api import CliBackend, load_settings
from pjm_api.cli_adapter import CLI_ARGV_SECRET_WARNING, mask_command
from pjm_api.cli_zip import install_cli_zip
from pjm_api.config import EXTENDED_OASIS_URLS
from pjm_api.doctor import format_doctor_report, run_doctor
from pjm_api.oasis_cli import qualify_result_with_criteria
from pjm_api.production import assert_production_action_allowed
from pjm_api.templates import get_template_info, list_templates

print("pjm-api imports ready")


## 2. Configure the notebook

For the normal persistent path, run `pjm-api cli install --dir ~/.pjm/cli` and `pjm-api init` in a terminal first. Then set `USE_SAVED_SETUP = True`. In a notebook, set `PJM_MASTER_PASSWORD` in the shell before launching Jupyter if you want the encrypted credentials file to unlock without a prompt.

If you are still learning or do not want the notebook to read saved credentials, leave `USE_SAVED_SETUP = False` and edit the values below. Blank values can also fall back to environment variables such as `PJM_USERNAME`, `PJM_PASSWORD`, `PJM_CERT_PATH`, `PJM_CERT_PASSWORD`, `PJM_CLI_JAR_PATH`, `PJM_CLI_JAVA_PATH`, and `PJM_CLI_DOWNLOADS`.

`INSTALL_PJM_CLI_JAR` is off by default. When enabled, it downloads PJM's CLI ZIP and extracts `pjm-cli.jar`; leave it off if you already have a local jar path or prefer to install the jar manually. If `JAR_PATH` and `PJM_CLI_JAR_PATH` are blank, `load_settings()` automatically uses `~/.pjm/cli/pjm-cli.jar` when that file exists.


In [ ]:
# ---- EDIT THESE VALUES ----
# True uses ~/.pjm/credentials.enc when it can be unlocked.
# False uses the explicit values in this cell plus environment variables.
USE_SAVED_SETUP = False

USERNAME = ""
PASSWORD = ""
CERTIFICATE_PATH = ""
CERTIFICATE_PASSWORD = ""

# Java executable and PJM's official CLI jar.
# Leave JAVA_PATH blank to use java from PATH or PJM_CLI_JAVA_PATH.
JAVA_PATH = ""
JAR_PATH = ""  # blank uses PJM_CLI_JAR_PATH, then ~/.pjm/cli/pjm-cli.jar if present
DOWNLOADS_DIR = "downloads"

# Optional jar download. Disabled by default so notebook execution does not fetch external data.
INSTALL_PJM_CLI_JAR = False
CLI_INSTALL_DIR = "~/.pjm/cli"
FORCE_JAR_INSTALL = False

# Public environments are TRAIN and PRODUCTION. TEST/STAGE require private PJM URLs.
ENVIRONMENT = "TRAIN"
OASIS_URL = ""
TEST_OASIS_URL = ""
STAGE_OASIS_URL = ""

# Network calls are explicit opt-ins later in the notebook.
TIMEOUT_SEC = 120
ALLOW_PRODUCTION_WRITE = False
DISABLE_PRODUCTION_WARNING = False
# --------------------------

DEFAULT_CLI_JAR_PATH = Path("~/.pjm/cli/pjm-cli.jar").expanduser()

if INSTALL_PJM_CLI_JAR:
    JAR_PATH = str(
        install_cli_zip(Path(CLI_INSTALL_DIR).expanduser(), force=FORCE_JAR_INSTALL)
    )

settings = load_settings(
    username=USERNAME,
    password=PASSWORD,
    certificate=CERTIFICATE_PATH,
    certificate_password=CERTIFICATE_PASSWORD,
    environment=ENVIRONMENT,
    oasis_url=OASIS_URL,
    backend="cli",
    java_path=JAVA_PATH,
    jar_path=JAR_PATH,
    downloads_dir=DOWNLOADS_DIR,
    timeout_sec=TIMEOUT_SEC,
    disable_production_warning=DISABLE_PRODUCTION_WARNING,
    allow_production_write=ALLOW_PRODUCTION_WRITE,
    use_credentials_file=USE_SAVED_SETUP,
    prompt_unlock=False,
)

print("known environments:", ", ".join(sorted(EXTENDED_OASIS_URLS)))
print("saved setup:", "enabled" if USE_SAVED_SETUP else "disabled")
print("environment:", settings.environment)
print("oasis_url:", settings.oasis_base_url)
print("backend:", settings.backend)
print("downloads:", settings.downloads_dir)
print("username:", settings.username or "(not set)")
print("password:", "***" if settings.password else "(not set)")
print("certificate:", settings.certificate_path or "(not set)")
print("default jar lookup:", DEFAULT_CLI_JAR_PATH)
print("jar_path:", settings.jar_path or "(not set)")
print("java_path:", settings.java_path)
print("note:", CLI_ARGV_SECRET_WARNING)


## 3. Understand Java and the jar

`pjm-api` defaults to the CLI backend because PJM's official browserless path is the Java jar. Python prepares a `PJMSettings` object, then `CliBackend` runs a subprocess shaped like this:

`java -Xms64m -Xmx256m -jar pjm-cli.jar -d downloads -u USER -p PASSWORD -r cert.p12|secret -s OASIS_URL -a ACTION -q KEY=VALUE -o outfile.txt -z 180000`

The password and certificate secret are masked when the command is displayed below.


In [ ]:
JAVA_AVAILABLE = False

try:
    java_probe = subprocess.run(
        [settings.java_path, "-version"],
        capture_output=True,
        text=True,
        timeout=15,
    )
    java_output = (java_probe.stderr or java_probe.stdout or "").strip()
    JAVA_AVAILABLE = java_probe.returncode == 0

    print("java executable:", settings.java_path)
    print("java found on PATH:", which(settings.java_path) or "(not found by PATH lookup)")
    print("java -version returncode:", java_probe.returncode)
    print(java_output or "(java produced no version output)")
except FileNotFoundError as exc:
    print("java executable:", settings.java_path)
    print("java -version failed:", exc)

if settings.jar_path:
    print("jar exists:", settings.jar_path.exists())
    print("jar path:", settings.jar_path)
else:
    print("jar path: (not set)")


## 4. Run local doctor checks

This uses the same doctor path as `pjm-api doctor --offline`. It checks configuration, the local certificate file, Java, and the local `pjm-cli.jar` without contacting PJM.


In [ ]:
missing = settings.missing_for_backend()
if missing:
    print("missing:", ", ".join(missing))

steps, offline_passed = run_doctor(settings, offline=True)
print(format_doctor_report(steps, offline_passed, offline=True))


## 5. Preview the TRAIN smoke command

This builds the TRANSSERV smoke command through `CliBackend`. No network call happens in this cell; it only shows the jar invocation that will be used.


In [ ]:
backend = None
smoke_cmd: list[str] = []
jar_ready = bool(settings.jar_path and settings.jar_path.exists())
LOCAL_READY = offline_passed and not missing and JAVA_AVAILABLE and jar_ready

if LOCAL_READY:
    backend = CliBackend(settings)
    smoke_cmd = backend.build_smoke_test_cmd(outfile=f"smoke_{settings.environment.lower()}.txt")
    print("masked command:")
    print(" ".join(mask_command(smoke_cmd)))
else:
    print("Fix local checks before building or running the PJM CLI command.")
    print("offline_passed:", offline_passed)
    print("missing:", ", ".join(missing) if missing else "none")
    print("java_available:", JAVA_AVAILABLE)
    print("jar_ready:", jar_ready)


## 6. Run the TRAIN smoke test

Set `RUN_SMOKE = True` only after the local doctor checks pass. A passing result means Java, `pjm-cli.jar`, credentials, the login certificate, the OASIS URL, and the TRANSSERV request parameters all work together.


In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    if backend is None or not smoke_cmd:
        raise RuntimeError("Run the preview cell successfully before the smoke test.")
    smoke_result = backend.run_cli(
        smoke_cmd,
        timeout_sec=TIMEOUT_SEC,
        print_results=True,
    )
    smoke_ok = qualify_result_with_criteria(
        smoke_result.returncode,
        smoke_result.stdout,
        smoke_result.stderr,
        code_equals=0,
        stdout_contains_all=["SUCCESS"],
        stderr_must_be_empty=True,
    )
    print(f"[SMOKE TEST:{settings.environment}] {'PASS' if smoke_ok else 'FAIL'}")
else:
    print("Smoke test skipped. Set RUN_SMOKE=True after local checks pass.")


## 7. Inspect templates before requesting them

Template metadata comes from the package catalog. Start with read-only templates such as `TRANSSERV`; input templates are guarded more strictly in production.


In [ ]:
for template in list_templates():
    print(f"{template.name:16} {template.type:8} {template.description}")

transserv = get_template_info("TRANSSERV")
print()
print("TRANSSERV metadata:")
print(transserv)


## 8. Optional read-only template request

This uses `CliBackend.run_template`, which builds and runs the same Java jar command pattern. Keep it disabled until the smoke test passes.


In [ ]:
RUN_TEMPLATE_REQUEST = False
TEMPLATE_NAME = "TRANSSERV"
TEMPLATE_PARAMS = {
    "OUTPUT_FORMAT": "DATA",
    "PRIMARY_PROVIDER_CODE": "PJM",
    "PRIMARY_PROVIDER_DUNS": "073647877",
    "RETURN_TZ": "EP",
    "VERSION": "3.3",
}

if RUN_TEMPLATE_REQUEST:
    if backend is None:
        raise RuntimeError("Run the preview cell successfully before template requests.")
    template_result = backend.run_template(
        template=TEMPLATE_NAME,
        outfile=f"{TEMPLATE_NAME.lower()}_{settings.environment.lower()}.txt",
        timeout_sec=TIMEOUT_SEC,
        params=TEMPLATE_PARAMS,
        print_results=True,
    )
    print("output file:", template_result.output_file)
else:
    print("Template request skipped. Set RUN_TEMPLATE_REQUEST=True after the smoke test passes.")


## 9. Optional TEST/STAGE loop

TEST and STAGE URLs are blank in the package because those environments are private. Fill `TEST_OASIS_URL` or `STAGE_OASIS_URL` in the setup cell before enabling this loop.


In [ ]:
RUN_PRIVATE_ENVIRONMENT_SMOKES = False
SMOKE_ENVIRONMENTS = ["TRAIN"]
PRIVATE_OASIS_URLS = {
    "TEST": TEST_OASIS_URL,
    "STAGE": STAGE_OASIS_URL,
}

if RUN_PRIVATE_ENVIRONMENT_SMOKES:
    for env in SMOKE_ENVIRONMENTS:
        env_settings = load_settings(
            username=settings.username,
            password=settings.password,
            certificate=str(settings.certificate_path or ""),
            certificate_password=settings.certificate_password or "",
            environment=env,
            oasis_url=PRIVATE_OASIS_URLS.get(env.upper(), ""),
            backend="cli",
            java_path=settings.java_path,
            jar_path=str(settings.jar_path or ""),
            downloads_dir=str(settings.downloads_dir),
            timeout_sec=TIMEOUT_SEC,
            disable_production_warning=DISABLE_PRODUCTION_WARNING,
            allow_production_write=ALLOW_PRODUCTION_WRITE,
            use_credentials_file=False,
            prompt_unlock=False,
        )
        print("=" * 72)
        print(f"Running smoke test for {env_settings.environment} ...")
        ok = CliBackend(env_settings).smoke_test(
            timeout_sec=TIMEOUT_SEC,
            outfile=f"smoke_{env_settings.environment.lower()}.txt",
        )
        print(f"[SMOKE TEST:{env_settings.environment}] {'PASS' if ok else 'FAIL'}")
else:
    print("Private environment smokes skipped.")


## 10. Production guard demonstration

This cell does not contact PJM. It shows that the package blocks production write or reservation-style actions unless you explicitly opt in.


In [ ]:
production_settings = load_settings(
    username=settings.username,
    password=settings.password,
    certificate=str(settings.certificate_path or ""),
    certificate_password=settings.certificate_password or "",
    environment="PRODUCTION",
    backend="cli",
    java_path=settings.java_path,
    jar_path=str(settings.jar_path or ""),
    downloads_dir=str(settings.downloads_dir),
    timeout_sec=TIMEOUT_SEC,
    disable_production_warning=True,
    allow_production_write=False,
    use_credentials_file=False,
    prompt_unlock=False,
)

try:
    assert_production_action_allowed(
        production_settings,
        method="PUT",
        template="pjmtransreq",
    )
except Exception as exc:
    print("Blocked as expected:", exc)


## What to do next

When the TRAIN smoke test passes, keep using the imported backend methods for reviewable, repeatable requests. Prefer TRAIN while learning the command shape. Move to PRODUCTION reads only after TRAIN passes, and leave production writes blocked unless you are intentionally performing a production action.

For a `.py` script after setup, keep it this small:

```python
from pjm_api import CliBackend, load_settings

backend = CliBackend(load_settings())
result = backend.run_template(
    template="TRANSSERV",
    params={"OUTPUT_FORMAT": "DATA"},
    outfile="transserv.txt",
)
print(result.returncode, result.output_file)
```

`load_settings()` reads the persistent setup: `~/.pjm/credentials.enc`, environment overrides, and the default jar at `~/.pjm/cli/pjm-cli.jar`. For non-interactive jobs, set `PJM_MASTER_PASSWORD` in that job's environment.
